<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Descarga del potencial exportador de Colombia desde el Export Potential Map del ITC

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas`: para manipulación y análisis de datos estructurados.
- `requests`: para consultar la interfaz del portal.
- `base64`, `json` y `cryptography`: para descifrar las respuestas.
- `time`: para espaciar las consultas.
- `tqdm`: barra de avance de la descarga por mercado.

In [1]:
import pandas as pd
import requests
import base64
import json
import time

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding as sym_padding
from cryptography.hazmat.backends import default_backend

from tqdm.auto import tqdm

/Users/juandiegoheredianino/Documents/nearshoring-colombia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, fecha_hoy

### Parámetros

Define el punto de acceso de la interfaz del Centro de Comercio Internacional, el código del país
exportador, el idioma de los nombres de mercado y la espera entre consultas.

El código 170 corresponde a Colombia en la nomenclatura interna del portal. La espera evita
saturar el servicio, que no publica un límite de tasa.

In [3]:
# Con la bandera en False el notebook no consulta el portal: reutiliza la descarga ya
# archivada en data/raw/itc/. Volver a bajar produce un archivo con la fecha del dia y
# cambia los resultados de toda la cadena aguas abajo, porque 01_cleaning/05_potencial_exportador
# toma siempre la descarga mas reciente.
reejecutar_descarga = False

BASE = 'https://exportpotential.intracen.org'

EXPORTADOR = 170   # Colombia en la nomenclatura del portal
ESPERA = 0.3       # segundos entre consultas

# el portal sirve los nombres de mercado en varios idiomas. Se piden en inglés porque
# así coinciden con los de ISO ITC Treemap.xlsx, que es lo que traduce a código ISO3
IDIOMA = 'en'

ENCABEZADOS = {
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'es-ES',
    'Referer': f'{BASE}/',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15',
    'X-Context': 'epm2',
}

carpeta_itc = raw / 'itc'

### Correspondencia entre códigos HS6 y grupos de producto del ITC

El portal agrupa los productos en categorías propias, distintas de la nomenclatura arancelaria.
La tabla de correspondencia permite traducir entre ambas y llega como un libro de Excel sin
cifrar.

In [4]:
if reejecutar_descarga:
    carpeta_itc.mkdir(parents=True, exist_ok=True)

    respuesta = requests.get(f'{BASE}/api/en/correspondences/full-table', headers=ENCABEZADOS, timeout=30)
    respuesta.raise_for_status()

    archivo_correspondencias = carpeta_itc / 'itc_correspondences.xlsx'
    archivo_correspondencias.write_bytes(respuesta.content)

    df_grupos_itc = pd.read_excel(archivo_correspondencias, dtype=str)
    df_grupos_itc['HS6'] = df_grupos_itc['HS6'].str.zfill(6)
    df_grupos_itc['k'] = df_grupos_itc['k'].str.zfill(4)

    print(f"{len(df_grupos_itc):,} correspondencias, {df_grupos_itc['HS6'].nunique():,} códigos HS6, {df_grupos_itc['k'].nunique():,} grupos")

### Descifrado de las respuestas

El portal devuelve los datos de potencial exportador cifrados con AES de 128 bits en modo CBC.
La clave viaja ofuscada en el JavaScript del sitio y se recupera sumando nueve al código de cada
carácter de una cadena fija; el vector de inicialización es de ceros.

La consulta conserva el manejo de la respuesta 406, que el servicio devuelve cuando la sesión no
se ha inicializado desde el navegador. Es la única condición que interrumpe la descarga.

In [5]:
# La clave viaja ofuscada en el JavaScript del sitio: la funcion dsafLGbwedjfk(str, 9)
# suma 9 al codigo de cada caracter. El vector de inicializacion es de ceros.
_CLAVE_OFUSCADA = "<,+-:/;=)./:;,0*('-/9,))<-0,;+=)"
CLAVE_AES = ''.join(chr(ord(c) + 9) for c in _CLAVE_OFUSCADA).encode('utf-8')
VECTOR_INICIAL = bytes(16)


def descifrar(texto_b64):
    """Descifra una respuesta AES-128-CBC de la API y la devuelve como objeto de Python."""
    cifrado = base64.b64decode(texto_b64)
    descifrador = Cipher(algorithms.AES(CLAVE_AES), modes.CBC(VECTOR_INICIAL),
                         backend=default_backend()).decryptor()
    con_relleno = descifrador.update(cifrado) + descifrador.finalize()
    sin_relleno = sym_padding.PKCS7(128).unpadder()
    return json.loads((sin_relleno.update(con_relleno) + sin_relleno.finalize()).decode('utf-8'))


def consultar(ruta):
    """Consulta un endpoint de la API y descifra la respuesta cuando viene cifrada."""
    r = requests.get(BASE + ruta, headers=ENCABEZADOS, timeout=20)
    if r.status_code == 406:
        raise RuntimeError('La API respondio 406: abre exportpotential.intracen.org en el navegador '
                           'para inicializar la sesion y vuelve a correr la celda.')
    r.raise_for_status()
    tipo = r.headers.get('Content-Type', '')
    return descifrar(r.text) if ('application/text' in tipo or 'text/plain' in tipo) else r.json()

### Mercados de destino

Recupera el catálogo de países para los que el portal publica estimaciones.

In [6]:
if reejecutar_descarga:
    lista_paises = consultar(f'/api/{IDIOMA}/countries')

    print(f"{len(lista_paises)} mercados disponibles")

### Potencial exportador por mercado

Consulta, para cada mercado de destino, los productos con potencial exportador desde Colombia.

**Pasos realizados:**

1. **Consulta por mercado:** obtiene la lista de productos con su potencial, las exportaciones observadas, el potencial ya realizado y la brecha.

2. **Registro de fallos:** los mercados que no responden se acumulan aparte en vez de interrumpir la descarga, y se exportan junto con el resultado.

3. **Espera entre consultas:** separa las peticiones según el parámetro definido arriba.

In [7]:
if reejecutar_descarga:
    registros = []
    errores = []

    for pais in tqdm(lista_paises, desc='Mercados de destino'):
        codigo, nombre = pais.get('code'), pais.get('name', '')
        try:
            datos = consultar(f'/api/{IDIOMA}/epis/products/from/i/{EXPORTADOR}/to/j/{codigo}/what/k/all')
        except Exception as e:
            errores.append({'market_code': codigo, 'market_name': nombre, 'error': str(e)})
            continue
        for p in datos if isinstance(datos, list) else []:
            item = p.get('item', {})
            registros.append({
                'market_code':        codigo,
                'market_name':        nombre,
                'product_code':       item.get('parent', {}).get('code'),
                'product_name':       item.get('parent', {}).get('name'),
                'hs6_code':           item.get('code'),
                'hs6_name':           item.get('name'),
                'export_potential':   p.get('value'),
                'current_exports':    p.get('exportValue'),
                'realized_potential': p.get('realizedPotential'),
                'gap':                p.get('gap'),
                'rank':               p.get('rank'),
            })
        time.sleep(ESPERA)

    df_potencial_itc = pd.DataFrame(registros)
    print(f"{len(df_potencial_itc):,} filas, {df_potencial_itc['market_code'].nunique()} mercados, {len(errores)} errores")

### Exportación

Guarda el potencial exportador y, si los hubo, el registro de mercados que fallaron, con la fecha
de descarga en el nombre del archivo.

Con `reejecutar_descarga` en `False` no se escribe nada: la celda reporta la descarga archivada que
va a consumir el resto de la cadena.

In [8]:
carpeta_itc.mkdir(parents=True, exist_ok=True)

if reejecutar_descarga:
    df_potencial_itc.to_parquet(carpeta_itc / f'colombia_export_potential_{fecha_hoy}.parquet', index=False)

    if errores:
        pd.DataFrame(errores).to_excel(carpeta_itc / f'errores_{fecha_hoy}.xlsx', index=False)

archivo_vigente = sorted(carpeta_itc.glob('colombia_export_potential_*.parquet'))[-1]
print(f"Descarga que consume la cadena: {archivo_vigente.name}")
print(f"{len(pd.read_parquet(archivo_vigente)):,} filas")

Descarga que consume la cadena: colombia_export_potential_2026-09-19.parquet
179,889 filas
